<a href="https://colab.research.google.com/github/arnavghai07/NLP/blob/main/Coreference_Resolution_using_neuralcoref.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
!pip install -q transformers==4.36.2 fastcoref

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.30.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
sentence-transformers 5.7.0 requires tokenizers>=0.19, but you have tokenizers 0.15.2 which is incompatible.
sentence-transformers 5.7.0 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.36.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.0 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [10]:
import sys
import transformers
from fastcoref import FCoref
from fastcoref.modeling import FCorefModel

if not hasattr(FCorefModel, 'all_tied_weights_keys'):
    FCorefModel.all_tied_weights_keys = {}

def resolve_coreferences(text_input: str) -> str:
    model = FCoref(device='cpu')
    predictions = model.predict(texts=[text_input])
    # Get character offsets instead of text strings
    clusters = predictions[0].get_clusters(as_strings=False)

    resolved_text = text_input
    replacements = []
    for cluster in clusters:
        main_mention_indices = cluster[0]
        start_main, end_main = main_mention_indices[0], main_mention_indices[1]
        main_mention_text = text_input[start_main:end_main]

        for mention_indices in cluster[1:]:
            start_ment, end_ment = mention_indices[0], mention_indices[1]
            replacements.append((start_ment, end_ment, main_mention_text))

    # Sort by starting index descending to replace from end to beginning
    replacements.sort(key=lambda x: x[0], reverse=True)
    for start, end, replacement in replacements:
        resolved_text = resolved_text[:start] + replacement + resolved_text[end:]
    return resolved_text

resolved_output = resolve_coreferences(unique_story_data)
print('\nOriginal:')
print(unique_story_data)
print('\nResolved:')
print(resolved_output)

Loading weights:   0%|          | 0/133 [00:00<?, ?it/s]

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

Inference:   0%|          | 0/1 [00:00<?, ?it/s]


Original:
Alice and her supervisor Dr. Sterling met in the laboratory. She handed a critical report to him, hoping he would approve it. Dr. Sterling skimmed the document quickly, and then he smiled and signed it.

Resolved:
Alice and Alice supervisor Dr. Sterling met in the laboratory. Alice handed a critical report to her supervisor Dr. Sterling, hoping her supervisor Dr. Sterling would approve it. her supervisor Dr. Sterling skimmed it quickly, and then her supervisor Dr. Sterling smiled and signed it.
